Model Creation

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
from scipy.stats import uniform



# Load the dataset
df = pd.read_csv('/Users/bapbap23/Desktop/Patient-No-Show-prediction-/patients.csv')

In [36]:
df.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [37]:
# 3. Convert 'No-show' to numerical
df['No-show_numeric'] = df['No-show'].apply(lambda x: 1 if x == 'Yes' else 0)

In [38]:
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])

In [39]:
df['ScheduledHour'] = df['ScheduledDay'].dt.hour
df['AppointmentHour'] = df['AppointmentDay'].dt.hour
df['ScheduledWeekday'] = df['ScheduledDay'].dt.dayofweek  # 0=Monday
df['AppointmentWeekday'] = df['AppointmentDay'].dt.dayofweek
df['AppointmentMonth'] = df['AppointmentDay'].dt.month

In [40]:
df['DaysBetween'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days
df['DaysBetween'] = df['DaysBetween'].clip(lower=0)

In [41]:
bins = [-1, 0, 7, 30, df['DaysBetween'].max()]
labels = ['0_days', '1-7_days', '8-30_days', '>30_days']
df['DaysBetween_binned'] = pd.cut(df['DaysBetween'], bins=bins, labels=labels, right=True)

In [42]:
df['SMS_DaysBetween'] = df['SMS_received'] * df['DaysBetween']
df['Age_Hipertension'] = df['Age'] * df['Hipertension']

In [43]:
neighbourhood_target_encoding = df.groupby('Neighbourhood')['No-show_numeric'].transform('mean')
df['Neighbourhood_Encoded'] = neighbourhood_target_encoding

In [44]:
df['Weekday_Diff'] = (df['AppointmentWeekday'] - df['ScheduledWeekday'] + 7) % 7

In [45]:
biased_features_final = ['Gender', 'Handcap', 'Diabetes', 'Scholarship', 'Age', 'DaysBetween', 'SMS_received',
                         'ScheduledWeekday', 'AppointmentWeekday', 'AppointmentHour', 'ScheduledHour', 'AppointmentMonth',
                         'DaysBetween_binned', 'SMS_DaysBetween', 'Age_Hipertension', 'Neighbourhood_Encoded', 'Weekday_Diff']

In [46]:
X_biased_final = df[biased_features_final]
y_biased_final = df['No-show_numeric']

In [47]:
categorical_features_to_encode_biased = ['Gender', 'ScheduledWeekday', 'AppointmentWeekday', 'AppointmentMonth', 'DaysBetween_binned']
X_biased_final = pd.get_dummies(X_biased_final, columns=categorical_features_to_encode_biased, drop_first=True)

In [48]:
X_train_biased_final, X_test_biased_final, y_train_biased_final, y_test_biased_final = train_test_split(X_biased_final, y_biased_final, test_size=0.2, random_state=42)

# 15. Identify numerical columns in biased training data
numerical_cols_biased_final = X_train_biased_final.select_dtypes(include=np.number).columns.tolist()

# 16. Scale biased numerical features
scaler_biased_final = StandardScaler()
X_train_biased_final_scaled = scaler_biased_final.fit_transform(X_train_biased_final[numerical_cols_biased_final])
X_test_biased_final_scaled = scaler_biased_final.transform(X_test_biased_final[numerical_cols_biased_final])

# 17. Apply SMOTE to biased training data
smote_biased = SMOTE(random_state=42)
X_train_biased_resampled_final, y_train_biased_resampled_final = smote_biased.fit_resample(X_train_biased_final_scaled, y_train_biased_final)

In [49]:
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'learning_rate': uniform(0.01, 0.1),
    'num_leaves': [20, 31, 40, 50],
    'max_depth': [-1, 10, 20, 30],
    'min_child_samples': [20, 30, 40, 50],
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
}

# 26. Initialize biased RandomizedSearchCV
random_search_biased_final = RandomizedSearchCV(lgb.LGBMClassifier(random_state=42),
                                          param_distributions=param_dist,
                                          n_iter=50,
                                          scoring='accuracy',
                                          cv=3,
                                          verbose=1,
                                          random_state=42,
                                          n_jobs=-1)

In [50]:
# 27. Fit biased RandomizedSearchCV
print("Starting RandomizedSearchCV for biased LightGBM model...")
random_search_biased_final.fit(X_train_biased_resampled_final, y_train_biased_resampled_final)

# 28. Get the best biased model
best_lgbm_biased_final = random_search_biased_final.best_estimator_
print("\nBest parameters for biased LightGBM model:")
print(random_search_biased_final.best_params_)

Starting RandomizedSearchCV for biased LightGBM model...
Fitting 3 folds for each of 50 candidates, totalling 150 fits
[LightGBM] [Info] Number of positive: 47026, number of negative: 47026
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.034988 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1830
[LightGBM] [Info] Number of data points in the train set: 94052, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 47026, number of negative: 47026
[LightGBM] [Info] Number of positive: 47026, number of negative: 47026
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017654 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1815
[LightGBM] [Info] Number of data points 

KeyboardInterrupt: 

In [ ]:
print("\nBest Biased LightGBM Model Evaluation (with SMOTE, all New Features, and Tuned Hyperparameters):")
y_pred_best_lgbm_biased_final = best_lgbm_biased_final.predict(X_test_biased_final_scaled)
print("Accuracy:", accuracy_score(y_test_biased_final, y_pred_best_lgbm_biased_final))
print("Classification Report:\n", classification_report(y_test_biased_final, y_pred_best_lgbm_biased_final))
print("Confusion Matrix:\n", confusion_matrix(y_test_biased_final, y_pred_best_lgbm_biased_final))